# PathSnap data processing




In [ ]:
from pathlib import Path
import networkx as nx
import numpy as np
import pandas as pd
DATA_ROOT=Path('/home/neeraj-r-rugi/ML_Hackathon/Student folder'); OUT=Path('/home/neeraj-r-rugi/ML_Hackathon/processed'); OUT.mkdir(exist_ok=True)
RADIUS_M=75.; GAP_S=2.; MAX_SPEED=70.; EARTH=6371000.
def trip_ids(root): return sorted(p.name for p in root.iterdir() if p.is_dir() and p.name.isdigit())
def read_file(p,k):
    s={'track':(['lon','lat','timestamp'],{'lon':'float64','lat':'float64','timestamp':'float64'}),'nodes':(['lon','lat'],{'lon':'float64','lat':'float64'}),'arcs':(['from_node','to_node'],{'from_node':'int64','to_node':'int64'}),'route':(['arc_id'],{'arc_id':'int64'})}[k]
    return pd.read_csv(p,sep=r'\s+',header=None,names=s[0],dtype=s[1],engine='python')
def manifest_row(d,split):
    tid=d.name; r={'split':split,'trip_id':tid,'schema_valid':True,'schema_errors':[]}; fs={}
    for k in ('track','nodes','arcs','route'):
        p=d/f'{tid}.{k}'; r[f'{k}_path']=str(p)
        if not p.exists(): r['schema_valid']=False; r['schema_errors'].append('missing '+k); continue
        try: fs[k]=read_file(p,k); r['n_'+k]=len(fs[k])
        except Exception as e: r['schema_valid']=False; r['schema_errors'].append(f'{k}: {e}')
    if 'track' in fs:
        t=fs['track']; r.update(start_time=t.timestamp.min(),end_time=t.timestamp.max(),non_increasing_times=int((t.timestamp.diff().dropna()<=0).sum()),missing_values=int(t.isna().sum().sum()))
    if 'arcs' in fs and 'nodes' in fs:
        n=len(fs['nodes']); a=fs['arcs']; bad=((a.from_node<0)|(a.from_node>=n)|(a.to_node<0)|(a.to_node>=n)).sum(); r['invalid_arc_node_refs']=int(bad); r['schema_valid'] &= not bool(bad)
    if 'route' in fs and 'arcs' in fs:
        bad=(~fs['route'].arc_id.between(0,len(fs['arcs'])-1)).sum(); r['invalid_route_arc_refs']=int(bad); r['schema_valid'] &= not bool(bad)
    r['schema_errors']='; '.join(r['schema_errors']); return r
def xy(lon,lat,lon0,lat0): return np.deg2rad(np.asarray(lon)-lon0)*EARTH*np.cos(np.deg2rad(lat0)),np.deg2rad(np.asarray(lat)-lat0)*EARTH
def prepare(t,n,a):
    lon0,lat0=n.lon.mean(),n.lat.mean(); t=t.copy(); t['gps_index']=np.arange(len(t)); t['dt']=t.timestamp.diff(); t['is_time_gap']=t.dt>GAP_S; t['is_duplicate_time']=t.dt.fillna(1)<=0; t['is_missing']=t[['lon','lat','timestamp']].isna().any(axis=1); t['x_m'],t['y_m']=xy(t.lon,t.lat,lon0,lat0); dx,dy=t.x_m.diff(),t.y_m.diff(); t['step_distance_m']=np.hypot(dx,dy); t['speed_mps']=t.step_distance_m.div(t.dt.where(t.dt>0)); t['speed_kmh']=t.speed_mps*3.6; t['heading_deg']=(np.degrees(np.arctan2(dx,dy))+360)%360; t['is_speed_outlier']=t.speed_mps>MAX_SPEED; t['quality_flag']=np.select([t.is_missing,t.is_duplicate_time,t.is_time_gap,t.is_speed_outlier],['missing','non_increasing_time','time_gap','speed_outlier'],default='ok'); n=n.copy(); n['node_id']=np.arange(len(n)); n['x_m'],n['y_m']=xy(n.lon,n.lat,lon0,lat0); a=a.copy(); a['arc_id']=np.arange(len(a)); a['from_x']=n.x_m.iloc[a.from_node].to_numpy(); a['from_y']=n.y_m.iloc[a.from_node].to_numpy(); a['to_x']=n.x_m.iloc[a.to_node].to_numpy(); a['to_y']=n.y_m.iloc[a.to_node].to_numpy(); dx,dy=a.to_x-a.from_x,a.to_y-a.from_y; a['length_m']=np.hypot(dx,dy); a['bearing_deg']=(np.degrees(np.arctan2(dx,dy))+360)%360; a['min_x'],a['max_x']=np.minimum(a.from_x,a.to_x),np.maximum(a.from_x,a.to_x); a['min_y'],a['max_y']=np.minimum(a.from_y,a.to_y),np.maximum(a.from_y,a.to_y); return t,n,a
def candidates(t,a):
    out=[]
    for p in t.itertuples():
        q=a[(a.min_x<=p.x_m+RADIUS_M)&(a.max_x>=p.x_m-RADIUS_M)&(a.min_y<=p.y_m+RADIUS_M)&(a.max_y>=p.y_m-RADIUS_M)]
        if q.empty: out.append({'gps_index':p.gps_index,'arc_id':-1,'distance_to_arc_m':np.nan,'candidate_rank':-1}); continue
        ax,ay,bx,by=[q[c].to_numpy() for c in ('from_x','from_y','to_x','to_y')]; vx,vy=bx-ax,by-ay; f=np.clip(np.divide((p.x_m-ax)*vx+(p.y_m-ay)*vy,vx*vx+vy*vy,out=np.zeros(len(q)),where=(vx*vx+vy*vy)>0),0,1); d=np.hypot(p.x_m-(ax+f*vx),p.y_m-(ay+f*vy)); keep=d<=RADIUS_M; q=q.loc[keep].copy(); q['distance_to_arc_m']=d[keep]; q['projection_fraction']=f[keep]; q['gps_index']=p.gps_index; q['candidate_rank']=np.arange(len(q)); out.extend(q[['gps_index','arc_id','length_m','bearing_deg','distance_to_arc_m','projection_fraction','candidate_rank']].to_dict('records'))
    return pd.DataFrame(out)
def process(root,split,tid):
    d=root/tid; t,n,a=prepare(read_file(d/f'{tid}.track','track'),read_file(d/f'{tid}.nodes','nodes'),read_file(d/f'{tid}.arcs','arcs')); c=candidates(t,a); o=OUT/split/tid; o.mkdir(parents=True,exist_ok=True); t.to_csv(o/'track_features.csv',index=False); n.to_csv(o/'nodes_features.csv',index=False); a.to_csv(o/'arcs_features.csv',index=False); c.to_csv(o/'candidate_features.csv',index=False); return {'split':split,'trip_id':tid,'gps_rows':len(t),'node_rows':len(n),'arc_rows':len(a),'candidate_rows':len(c),'unmatched_gps':int((c.groupby('gps_index').arc_id.max()<0).sum())}
manifest=pd.DataFrame([manifest_row(root/t,split) for split,root in (('train',DATA_ROOT/'train'),('test',DATA_ROOT/'student_test')) for t in trip_ids(root)]); manifest.to_csv(OUT/'trip_manifest.csv',index=False); display(manifest.head()); display(pd.Series(process(DATA_ROOT/'train','train','00000001')))


,split,trip_id,schema_valid,schema_errors,track_path,n_track,nodes_path,n_nodes,arcs_path,n_arcs,route_path,n_route,start_time,end_time,non_increasing_times,missing_values,invalid_arc_node_refs,invalid_route_arc_refs
0,train,00000001,True,,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,1315,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,88809,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,165790,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,525,1.0,1358.000,0,0,0,0
1,train,00000002,True,,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,7243,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,7936,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,16139,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,262,1.0,8934.156,0,0,0,0
2,train,00000003,True,,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,821,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,9138,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,18028,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,79,1.0,824.000,0,0,0,0
3,train,00000004,True,,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,1064,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,7350,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,15499,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,320,1.0,5218.000,0,0,0,0
4,train,00000005,True,,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,1045,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,898,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,1579,/home/neeraj-r-rugi/ML_Hackathon/Student folde...,105,1.0,1045.000,0,0,0,0


split                train
trip_id           00000001
gps_rows              1315
node_rows            88809
arc_rows            165790
candidate_rows       28766
unmatched_gps            0
dtype: object

## Complete processing run, This is the production run thingy basically
 processes all discovered trips independently. A failed trip is recorded in `processing_summary.csv` and does not prevent other trips from being processed. For training trips, candidate rows are marked when their `arc_id` appears in the supplied reference route; this is a route-membership training feature, not a row-wise GPS/route alignment.


In [ ]:
def add_route_training_labels(split: str, trip_id: str) -> None:
    """Add route membership and nearest candidate labels for a processed trip."""
    trip_dir = OUT / split / trip_id
    route_path = DATA_ROOT / split / trip_id / f"{trip_id}.route"
    candidate_path = trip_dir / "candidate_features.csv"
    if not route_path.exists() or not candidate_path.exists():
        return

    route = read_file(route_path, "route")
    candidates = pd.read_csv(candidate_path)
    route_arc_ids = set(route["arc_id"].astype(int))
    candidates["is_reference_route_arc"] = candidates["arc_id"].isin(route_arc_ids)

    valid = candidates[candidates["is_reference_route_arc"] & candidates["arc_id"].ge(0)].copy()
    if valid.empty:
        labels = pd.DataFrame(columns=["gps_index", "reference_arc_id", "reference_distance_m"])
    else:
        best = valid.sort_values(["gps_index", "distance_to_arc_m"]).drop_duplicates("gps_index")
        labels = best[["gps_index", "arc_id", "distance_to_arc_m"]].rename(
            columns={"arc_id": "reference_arc_id", "distance_to_arc_m": "reference_distance_m"}
        )

    candidates = candidates.merge(labels, on="gps_index", how="left")
    candidates["is_reference_candidate"] = candidates["arc_id"].eq(candidates["reference_arc_id"])
    candidates.to_csv(candidate_path, index=False)
    route.assign(route_position=np.arange(len(route), dtype=np.int64)).to_csv(
        trip_dir / "route.csv", index=False
    )
    labels.to_csv(trip_dir / "gps_reference_labels.csv", index=False)


all_summaries = []
for split, root in (("train", DATA_ROOT / "train"), ("test", DATA_ROOT / "student_test")):
    for trip_id in trip_ids(root):
        try:
            summary = process(root, split, trip_id)
            add_route_training_labels(split, trip_id)
            summary["status"] = "processed"
        except Exception as exc:
            summary = {
                "split": split,
                "trip_id": trip_id,
                "status": "failed",
                "error_type": type(exc).__name__,
                "error": str(exc),
            }
            warnings.warn(f"Failed to process {split}/{trip_id}: {exc}")
        all_summaries.append(summary)

processing_summary = pd.DataFrame(all_summaries).sort_values(["split", "trip_id"])
processing_summary.to_csv(OUT / "processing_summary.csv", index=False)

expected = manifest[["split", "trip_id"]].drop_duplicates()
actual = processing_summary[["split", "trip_id"]].drop_duplicates()
coverage = expected.merge(actual, on=["split", "trip_id"], how="left", indicator=True)
missing_outputs = coverage.loc[coverage["_merge"] != "both", ["split", "trip_id"]]
failed = processing_summary[processing_summary["status"] != "processed"]

print("Expected trips:", len(expected))
print("Processed trips:", int((processing_summary["status"] == "processed").sum()))
print("Failed trips:", len(failed))
print("Missing summary rows:", len(missing_outputs))
display(processing_summary.groupby(["split", "status"]).size().rename("trip_count"))
display(failed)

if len(missing_outputs) or len(failed):
    raise RuntimeError("The complete preprocessing run did not process every trip; inspect processing_summary.csv.")

print(f"Complete preprocessing finished. Outputs are under: {OUT}")
